# Strategy 2 on the Q20 curve — the deep packs Citi actually traded

Strategy 2 as shipped reaches pack ranks 2–10, first expiry ≤ ~2.4y. Citi's
published SOFR screen is ranks **5–17**, and the two packs it recommended by
name are **Blues** (rank 13, T1 ≈ 3.25y) and **Golds** (rank 17, T1 ≈ 4.25y):

> "Sell $100k DV01 of **Blues** convexity adjustment, i.e. buy 1000 of H0-Z0
> packs (1000 of each of the four contracts) and pay $1bn on a
> matched-maturity (3/18/20-3/17/21) CME swap."

> "We continue to favor shorting **Blues** convexity adjustment as a short vol
> proxy." — *Rates Vol Lab*, 12-Jun-2023, p.14

So the part of the strategy the research is actually about had never been
tested here. This notebook is that test, and its two jobs are (1) to establish
**where a curve-derived futures rate is allowed to stand in for a settlement
mark**, and (2) to run the deep-pack backtest and put it next to the near-pack
result.

## The five findings, up front

1. **The premise in the brief is empirically false, and the reason matters.**
   There is no `USD-SOFR-1D-Q20STIRT` curve store on this machine at all
   (`curve_store/raw` carries Q12STIRT with 2,061 dates and Q16STIRT with 197).
   Worse, `IRSwapsMDP.get_pricer` for that curve makes **52–57 Barchart
   requests per date**, because the production builder's pricer fetcher is
   wired to `BARCHART_TOS_LIVE_STIRF-RL` — the *intraday* source, which reaches
   depth 20 on **zero** local dates. The module therefore builds the curve
   itself from the **17:00 EOD** source and injects the pricers into the
   production solver. Measured: **0 outbound requests, ~0.15s per date.**

2. **The earlier rejection of a curve-derived rate was right about the front
   end and wrong about the deep end.** That work measured a 9.375bp median
   disagreement in 2019 — *pooled across ranks*. Decomposed by rank it is
   **15.8bp at ranks 1–4 and 0.26bp at ranks 13–16**. The cause is now
   identified exactly: every `*STIRT` node grid is built from the central-bank
   meeting-date map, and this machine's map **starts in April 2021**. Before
   then the *front* of the curve is one enormous log-linear segment while the
   deep windows sit in a properly resolved region. **The 2019 degeneracy is a
   front-end defect.**

3. **The naive quality criteria select for the failure.** In 2019 the
   degenerate curve scores *better* on both `CA ≥ 0` and `corr(CA, T1²)`,
   because a flattened curve manufactures a monotone non-negative profile out
   of nothing. The gate here is built on node resolution and per-contract
   settle agreement instead, and `CA ≥ 0` is **reported but never gated on**.

4. **Gated, the Q20 forward *is* the settlement mark.** On gate-passed deep
   rows, median |CA(Q20) − CA(settle)| = **0.057bp** and the correlation is
   **0.9995**; at rank 13 it is 0.028bp and 1.0000. The 6/9/2023 tie-out
   reproduces all 13 Citi rows including **Blues 15.40 → 15.72 (+0.32bp)** and
   **Golds 22.29 → 21.73 (−0.56bp)**.

5. **And that is the case for trading deep packs.** The settle-timing noise
   barely grows with maturity while the adjustment grows four-fold: noise/CA
   level is **30.1% at ranks 1–8** and **6.7% at ranks 13–17**, so the
   CA-implied vol error falls from **15.1% to 3.4%**.

What it does **not** buy: new coverage. The curve is calibrated to the settles,
so it cannot conjure a contract the store lacks, and it *inherits* a stale
settle rather than laundering it. The binding constraint is the raw SR3
diskcache throughout.

In [1]:
import os

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

import dataclasses
import datetime
import math
import pathlib
import sys
import time

import numpy as np
import pandas as pd

_REPO = (pathlib.Path(__file__).resolve().parents[3] if "__file__" in dir()
         else pathlib.Path.cwd().parents[2])
sys.path.insert(0, str(_REPO))

import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

from BT.trade_dashboard import compare_curves, summary_stats, trade_dashboard

import RVUtils.ConvexityRV.strat2_q20 as Q
import RVUtils.ConvexityRV.strat2_sofr_convexity as S2
from RVUtils.ConvexityRV import ca_staleness
from RVUtils.ConvexityRV.ca_diagnostics import vol_sensitivity_bp_per_bp
from RVUtils.ConvexityRV.holee import implied_vol_from_ca_bp
from RVUtils.ConvexityRV.listed_cache_guard import cache_only, network_calls_blocked
from RVUtils.ConvexityRV.packs import imm_date, quarterly_imm_sequence

DATA = _REPO / "notebooks" / "data" / "convexity_rv"
DATA.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220)

C:\Users\chris\clee\ARBS-fix\RVUtils\ConvexityRV\curve_ops.py:61: LicenceNotice:


Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence



## 1. CONFIG — every knob, with why it is set where it is

Two config objects, deliberately separate. `Q20Config` describes **how a
futures rate is obtained and when it may be believed**; `Strat2Config`
describes **the screen and the trade** and is unchanged from the near-pack
strategy. `deep_pack_config` derives the second from the first so the pack
universe and the strip depth cannot drift apart.

In [2]:
@dataclasses.dataclass(frozen=True)
class NotebookConfig:
    """Everything this notebook chooses, over and above the module defaults."""

    # ---- the deep universe actually backtested -------------------------------
    deep_rank_start: int = 9
    """First RANKED pack window. Window ``rank_start-1`` is still needed (the 3m
    roll of the first ranked pack is measured against it), so the panel is
    filtered from ``rank_start-1``."""

    deep_n_packs: int = 6
    """Ranked windows 9..14 — Greens through one past Blues. Chosen by DATA, not
    by preference: windows 9..14 need 17 contracts and give the longest
    contiguous gate-passed run in the store (743 days). Citi's full 5..17 needs
    20 contracts and its longest run is 192 days, entirely inside ZIRP."""

    # ---- the near-pack comparison -------------------------------------------
    near_rank_start: int = 2
    near_n_packs: int = 9
    """Windows 2..10 — the shipped near-pack strategy, re-run on this panel and
    this gate so the comparison is like-for-like on pipeline as well as window."""

    # ---- the published reference --------------------------------------------
    citi_date: datetime.date = datetime.date(2023, 6, 9)
    citi_rank_start: int = 5
    citi_n_packs: int = 13
    """Citi Figure 58: windows 5..17, Reds M4-H5 through Golds M7-H8."""

    # ---- tolerances used only for the notebook's own assertions --------------
    tieout_max_abs_bp: float = 4.0
    """Per-row |ours − Citi| the tie-out asserts. The published near-pack tie-out
    on the same date was |max| 3.12bp against 13 rows, so 4.0 is that measured
    spread with headroom, not a target."""

    tieout_min_corr: float = 0.95
    iv_ratio_target: float = 0.9973
    """Inverting Citi's OWN CA column reproduces Citi's OWN implied-vol column at
    this median ratio. Uses Citi's numbers on both sides, so it validates the
    Ho-Lee form and the T1 convention with no market data involved."""


NB = NotebookConfig()
QCFG = Q.Q20Config()
DEEP = Q.deep_pack_config(rank_start=NB.deep_rank_start, n_packs=NB.deep_n_packs)
NEAR = Q.deep_pack_config(rank_start=NB.near_rank_start, n_packs=NB.near_n_packs)
CITI = Q.deep_pack_config(rank_start=NB.citi_rank_start, n_packs=NB.citi_n_packs)

for _n, _c in (("deep", DEEP), ("near", NEAR), ("citi", CITI)):
    print(f"{_n:5s}: windows {_c.rank_start}..{_c.rank_start + _c.n_packs - 1}, "
          f"n_contracts={_c.n_contracts}")
print(f"\ngate: max |settle - Q20 fwd| {QCFG.gate_max_settle_diff_bp}bp per CONTRACT, "
      f"min nodes inside {QCFG.gate_min_nodes_inside}, "
      f"min control power {QCFG.gate_min_control_power_bp}bp")

deep : windows 9..14, n_contracts=17
near : windows 2..10, n_contracts=13
citi : windows 5..17, n_contracts=20

gate: max |settle - Q20 fwd| 2.0bp per CONTRACT, min nodes inside 1, min control power 1.0bp


## 2. Sign probe — asserted, not assumed

The convexity adjustment is `pack_rate − matched_forward_swap_rate`, and it is
**positive** when the futures rate exceeds the forward — the no-arbitrage
direction. This runs the shipped `ca_snapshot` against a curve pinned at a
known rate, so the sign checked is the one the production code produces.

In [3]:
class _PinnedCurve:
    """A swap curve whose matched forward rate is always 4.00%."""

    _rl_curve_handle = None

    def _curve_definition(self):
        return {"ReferenceRate": "usd_irs"}


_orig_par = S2._swap_par_rate
try:
    S2._swap_par_rate = lambda *a, **k: 4.00
    _seq = quarterly_imm_sequence(NB.citi_date, CITI.n_contracts)
    for _fut_rate, _want in ((4.10, +10.0), (3.90, -10.0), (4.00, 0.0)):
        _snap = S2.ca_snapshot(NB.citi_date, CITI,
                               futures_prices=Q.forwards_to_prices({k: _fut_rate for k in _seq}),
                               swap_pricer=_PinnedCurve())
        assert len(_snap) > 0, "sign probe produced no rows"
        assert np.allclose(_snap["ca_bp"].to_numpy(float), _want, atol=1e-9), (
            f"futures {_fut_rate}% vs swap 4.00% should give {_want:+.1f}bp, "
            f"got {_snap['ca_bp'].unique()}")
        print(f"  futures {_fut_rate:.2f}% - swap 4.00%  ->  CA {_want:+6.1f}bp  OK "
              f"({len(_snap)} windows)")
finally:
    S2._swap_par_rate = _orig_par

# futures DV01 is rate-invariant and identical for ED and SR3
from RVUtils.ConvexityRV.packs import DV01_PER_CONTRACT

assert DV01_PER_CONTRACT == 25.0
assert int(round(100_000.0 / (4.0 * DV01_PER_CONTRACT))) == 1000, "1000 packs = $100k/bp"
print(f"  $100k DV01 = {int(round(100_000.0/(4*DV01_PER_CONTRACT)))} packs "
      f"x 4 legs x ${DV01_PER_CONTRACT:.0f}/bp  OK")

# roll-date off-by-one: the SFRCM ladder rolls, the pack universe does not
_roll = imm_date(2019, 6)
assert Q.is_imm_roll_date(_roll) and Q.instrument_count(_roll, 20) == 19
assert Q.instrument_count(NB.citi_date, 20) == 20
print(f"  IMM roll {_roll}: strip depth 20 -> {Q.instrument_count(_roll, 20)} SFRCM "
      f"instruments  OK")

  futures 4.10% - swap 4.00%  ->  CA  +10.0bp  OK (17 windows)
  futures 3.90% - swap 4.00%  ->  CA  -10.0bp  OK (17 windows)
  futures 4.00% - swap 4.00%  ->  CA   +0.0bp  OK (17 windows)
  $100k DV01 = 1000 packs x 4 legs x $25/bp  OK
  IMM roll 2019-06-19: strip depth 20 -> 19 SFRCM instruments  OK


## 3. Universe — measured off the diskcache, not assumed

`strip_depth_by_date` reads the SR3 store's sqlite shards read-only and returns
the length of the **contiguous** front strip per date. Contiguity is required
rather than mere presence: a pack is four consecutive contracts and the curve
calibrates to an unbroken `SFRCM1..k` prefix, so "17 of the first 20 present"
would overstate what is buildable.

In [4]:
_t0 = time.time()
DEPTHS = Q.strip_depth_by_date(QCFG)
print(f"{len(DEPTHS)} dates {min(DEPTHS)}..{max(DEPTHS)}  ({time.time()-_t0:.0f}s)")

_d = pd.Series(DEPTHS, name="depth")
_d.index = pd.to_datetime(list(DEPTHS))
_u = pd.DataFrame({"n_dates": _d.groupby(_d.index.year).size()})
for _r in (5, 9, 13, 17):
    _u[f"rank{_r}"] = (_d >= Q.depth_for_rank(_r)).groupby(_d.index.year).sum()
_u.loc["TOTAL"] = _u.sum()
print("\ndates able to quote each pack rank (rank 13 = Blues, 17 = Golds):")
print(_u.to_string())
assert int(_u.loc["TOTAL", "rank13"]) > 1000, "Blues should be reachable on >1000 dates"
assert int(_u.loc["TOTAL", "rank17"]) > 500, "Golds should be reachable on >500 dates"

2097 dates 2018-05-04..2026-08-19  (26s)

dates able to quote each pack rank (rank 13 = Blues, 17 = Golds):
       n_dates  rank5  rank9  rank13  rank17
2018       167    167    167     167      41
2019       253    253    252     252      77
2020       253    253    253     253     253
2021       252    252    252     252     187
2022       252    252    252     195      52
2023       258    258    231      51      51
2024       252    251    249     248     248
2025       251    250    249     248     247
2026       159    159    159     157     156
TOTAL     2097   2095   2064    1823    1312


## 4. The panel

Built by `scripts/strat2_q20_build.py` (6 workers, ~2 min for 1,410 dates).
Every row carries **both** rate sources, their two adjustments, the three gate
conditions and every measurement the gate was derived from — the gate travels
*with* the panel so a filter applied in one notebook is not a filter the next
reader silently omits.

In [5]:
PANEL = pd.read_parquet(DATA / "strat2_q20_panel.parquet")
RATES = pd.read_parquet(DATA / "strat2_q20_rates.parquet")
SETTLES = pd.read_parquet(DATA / "strat2_q20_settles.parquet")
PANEL["year"] = pd.to_datetime(PANEL["date"]).dt.year
PANEL["band"] = pd.cut(PANEL["rank"], [0, 4, 8, 12, 16, 20],
                       labels=["1-4", "5-8", "9-12", "13-16", "17-20"])
print(f"panel {PANEL.shape[0]:,} rows x {PANEL['date'].nunique():,} dates "
      f"{PANEL['date'].min().date()}..{PANEL['date'].max().date()}, "
      f"ranks {PANEL['rank'].min()}..{PANEL['rank'].max()}")
assert PANEL["rank"].max() >= 17, "the panel must reach Golds"
print(f"rows reaching Blues (rank 13): {int((PANEL['rank']==13).sum()):,} on "
      f"{PANEL[PANEL['rank']==13]['date'].nunique():,} dates")
print(f"rows reaching Golds (rank 17): {int((PANEL['rank']==17).sum()):,} on "
      f"{PANEL[PANEL['rank']==17]['date'].nunique():,} dates")

panel 32,540 rows x 2,068 dates 2018-05-04..2026-08-18, ranks 1..17
rows reaching Blues (rank 13): 1,807 on 1,807 dates
rows reaching Golds (rank 17): 1,302 on 1,302 dates


## 5. Node resolution — the test the zero-convexity control cannot do

`CA_synthetic` compares the arithmetic mean of four quarterly forwards with the
par rate of the swap spanning them. If the curve has **no node inside the
window**, log-linear interpolation makes all four forwards identical and the
two agree *by construction*: the control returns exactly 0.00bp however wrong
the leg is. A passing control on a flat segment is not evidence, it is an
absence of evidence, and it looks identical.

`spans_window` is that condition. Read the table by row and by column: the
defect is **entirely in the near ranks and entirely before 2021**, which is
when the central-bank meeting map begins.

In [6]:
_res = PANEL.groupby("year").agg(
    n_dates=("date", "nunique"), q20_nodes_med=("q20_n_nodes", "median"),
    nodes_inside_med=("q20_n_nodes_inside", "median"),
    segment_days_med=("q20_segment_days", "median"))
print("Q20 node grid by year:")
print(_res.to_string())

print("\nspans_window %% — a single node interval swallows the whole pack window:")
_sw = (PANEL.pivot_table(index="year", columns="band", values="q20_spans_window",
                         aggfunc="mean", observed=True) * 100).round(1)
print(_sw.to_string())
assert _sw.loc[2019, "1-4"] > 50, "2019 near packs must be degenerate"
assert float(_sw.loc[2019, "13-16"]) == 0.0, "2019 deep packs must be resolved"

Q20 node grid by year:
      n_dates  q20_nodes_med  nodes_inside_med  segment_days_med
year                                                            
2018      163           21.0               4.0              56.0
2019      250           28.0               7.0              49.0
2020      251           36.0               7.0              49.0
2021      250           41.0               7.0              49.0
2022      249           41.0               7.0              42.0
2023      249           39.0               7.0              42.0
2024      250           36.0               7.0              49.0
2025      249           32.0               5.0              91.0
2026      157           29.0               3.0              91.0

spans_window %% — a single node interval swallows the whole pack window:
band    1-4   5-8  9-12  13-16  17-20
year                                 
2018  100.0  68.7   0.0    0.0    0.0
2019   78.9   5.4   0.0    0.0    0.0
2020    5.3   0.0   0.0    0.0    0.

In [7]:
_fig = go.Figure()
for _b in ["1-4", "5-8", "9-12", "13-16", "17-20"]:
    if _b in _sw.columns:
        _fig.add_trace(go.Scatter(x=_sw.index, y=_sw[_b], mode="lines+markers", name=f"ranks {_b}"))
_fig.update_layout(
    title="Node-resolution failure is a FRONT-END defect, not a deep-end one<br>"
          "<sub>share of pack-days whose whole window sits inside one Q20 node "
          "interval — the failure the zero-convexity control cannot see</sub>",
    xaxis_title="year", yaxis_title="% of pack-days degenerate", height=430,
    legend_title="pack rank")
_fig

## 6. Settle agreement — "ensure we correctly get the settlement marks"

The direct test. For every pack window, the **maximum over its four contracts**
of |SR3 settle − Q20 IMM forward|. Max, not mean: two legs wrong by +6bp and
two by −6bp average to a clean-looking pack.

This is the decomposition of the earlier 9.375bp-in-2019 rejection.

In [8]:
print("median max|settle - Q20 fwd| (bp), by year x rank band:")
_ag = PANEL.pivot_table(index="year", columns="band", values="max_settle_diff_bp",
                        aggfunc="median", observed=True).round(3)
print(_ag.to_string())
print("\np95 of the same:")
print(PANEL.pivot_table(index="year", columns="band", values="max_settle_diff_bp",
                        aggfunc=lambda s: s.quantile(.95), observed=True).round(3).to_string())
print(f"\n2019, ranks 1-4 : {_ag.loc[2019,'1-4']:.2f}bp  <- the pooled 9.375bp rejection "
      f"lives here")
print(f"2019, ranks 13-16: {_ag.loc[2019,'13-16']:.2f}bp  <- and NOT here")
assert _ag.loc[2019, "1-4"] > 5.0 and _ag.loc[2019, "13-16"] < 1.0

median max|settle - Q20 fwd| (bp), by year x rank band:
band     1-4     5-8   9-12  13-16  17-20
year                                     
2018  17.058  12.852  3.907  0.280  0.281
2019  15.765  10.004  0.268  0.258  0.549
2020   0.691   0.354  0.226  0.309  0.370
2021   0.298   0.317  0.416  0.478  2.195
2022   1.542   0.629  0.398  0.307  0.788
2023   1.641   0.755  0.348  0.646  1.900
2024   0.768   0.286  0.242  0.872  2.626
2025   0.772   0.330  0.362  1.168  4.009
2026   0.796   0.506  0.297  1.119  3.720

p95 of the same:
band     1-4     5-8    9-12  13-16  17-20
year                                      
2018  49.710  24.236  25.386  1.273  0.282
2019  37.641  28.175   2.273  0.549  0.952
2020  16.531   1.416   0.606  0.497  0.697
2021   0.941   0.674   0.688  2.050  4.258
2022   6.120   1.223   1.083  1.065  1.944
2023   3.174   1.781   0.800  2.060  3.943
2024   2.997   0.760   0.562  1.579  3.661
2025   1.728   0.627   0.594  2.003  4.909
2026   1.393   0.880   0.546  1.88

### The admissibility rule, and why it is set where it is

`gate_max_settle_diff_bp = 2.0`. Not a round number chosen for looks: Barchart's
"EOD" is the 1-minute bar nearest 17:00 New York while SR3 settles at ~15:00
ET, and the CA noise that mismatch induces was measured on near packs at
**0.89–2.55bp per pack-day (mean 1.82)**. A tolerance below that rejects rows
for carrying noise the settles themselves carry.

In [9]:
print("gate incidence by year (share of rows passing each condition):")
print(Q.gate_summary(PANEL).round(4).to_string())
print("\ngate pass rate %% by year x rank band:")
_gp = (PANEL.pivot_table(index="year", columns="band", values="gate_ok",
                         aggfunc="mean", observed=True) * 100).round(1)
print(_gp.to_string())
GATED = Q.apply_gate(PANEL)
print(f"\n{len(GATED):,} of {len(PANEL):,} rows pass ({100*len(GATED)/len(PANEL):.1f}%)")
assert _gp.loc[2019, "1-4"] < 5 and _gp.loc[2019, "13-16"] > 95, (
    "the gate must invert between near and deep packs in 2019")

gate incidence by year (share of rows passing each condition):
      gate_covered  gate_resolved  gate_settle_agrees  gate_ok  n_rows  n_dates
year                                                                           
2018           1.0         0.5719              0.3515   0.3398    2649      163
2019           1.0         0.7926              0.5470   0.5465    4075      250
2020           1.0         0.9684              0.9295   0.9109    4267      251
2021           1.0         0.9914              0.9634   0.9548    4178      250
2022           1.0         0.9849              0.8854   0.8703    3516      249
2023           1.0         0.9946              0.8652   0.8598    2797      249
2024           1.0         0.9746              0.9256   0.9002    4209      250
2025           1.0         1.0000              0.9295   0.9295    4200      249
2026           1.0         0.9826              0.9370   0.9196    2649      157

gate pass rate %% by year x rank band:
band   1-4    5-8

## 7. The vanity metrics — reported, and deliberately NOT gated on

The earlier investigation established that `CA ≥ 0` and `corr(CA, T1²)` **do
not discriminate**: in 2019 the degenerate curve scored *better* on both,
because a flattened curve manufactures a monotone non-negative profile out of
nothing. Building a gate on them selects for the failure it is meant to catch.
They are reported here so a reader can see they carry no signal.

In [10]:
_neg = pd.DataFrame({
    "CA_q20<0 %": PANEL.groupby("year")["ca_bp_q20"].apply(lambda s: 100 * (s < 0).mean()),
    "CA_settle<0 %": PANEL.groupby("year")["ca_bp_settle"].apply(lambda s: 100 * (s < 0).mean()),
    "gated CA_q20<0 %": GATED.groupby("year")["ca_bp_q20"].apply(lambda s: 100 * (s < 0).mean()),
}).round(1)
print(_neg.to_string())
print("\nBoth sources violate CA>=0 at essentially the SAME rate, and gating barely")
print("moves it: the violations are a property of the MARKET DATA (ZIRP, where the")
print("adjustment is ~0 and its sign is noise), not of the rate source. A gate on")
print("CA>=0 would therefore discard good deep rows and keep degenerate 2019 ones.")

      CA_q20<0 %  CA_settle<0 %  gated CA_q20<0 %
year                                             
2018        14.9            8.0               1.4
2019        36.0           42.7              38.6
2020        64.2           65.0              67.4
2021        31.9           32.4              33.0
2022         7.6            5.9               4.9
2023         4.1            3.3               2.7
2024         7.3            7.7               6.4
2025        20.8           22.2              22.0
2026        16.7           17.4              17.9

Both sources violate CA>=0 at essentially the SAME rate, and gating barely
moves it: the violations are a property of the MARKET DATA (ZIRP, where the
adjustment is ~0 and its sign is noise), not of the rate source. A gate on
CA>=0 would therefore discard good deep rows and keep degenerate 2019 ones.


## 7b. The zero-convexity control on the deep windows — and its blind spot

Replace the four futures rates with the **swap curve's own IMM×IMM forwards**.
Those carry no convexity by construction — they come off the very discount
curve the swap leg is priced on — so the adjustment must be ~0. Whatever it
returns instead is our own convention error, measured with no external
reference. The matched swap is **quarterly/quarterly** throughout; the
`usd_irs` spec default is annual fixed and biases every adjustment by
`0.375·r²`, which is up to 12bp and the same order as the signal.

**But the control must be read against the curvature it was allowed to see.**
`CA_synthetic` is an arithmetic-mean-minus-annuity-weighted-mean discrepancy,
and that is identically zero when the four forwards are identical. This is
where deep packs are *weaker* than near ones: `USD-SOFR-1D` thins toward annual
nodes past ~2y, so a deep window contains about **one** node and its four
quarterly forwards barely differ.

In [11]:
_ctl = PANEL.groupby("rank").agg(
    n=("ca_synthetic_bp", "size"),
    ca_syn_med=("ca_synthetic_bp", "median"),
    ca_syn_p95=("ca_synthetic_bp", lambda s: float(np.nanpercentile(np.abs(s), 95))),
    pred_med=("ca_synthetic_pred_bp", "median"),
    swap_spread_bp=("swap_fwd_spread_bp", "median"),
    swap_nodes_in=("swap_n_nodes_inside", "median"),
    annual_gap_bp=("annual_qq_gap_bp", "median"))
_ctl["resid_after_pred"] = (
    PANEL.assign(r=PANEL["ca_synthetic_bp"] - PANEL["ca_synthetic_pred_bp"])
    .groupby("rank")["r"].median())
print("zero-convexity control by rank (bp):")
print(_ctl.round(4).to_string())

_all = PANEL["ca_synthetic_bp"].to_numpy(float)
print(f"\nall {len(_all):,} rows: mean {np.nanmean(_all):+.4f}bp  median "
      f"{np.nanmedian(_all):+.4f}bp  |p95| {np.nanpercentile(np.abs(_all),95):.4f}bp")
assert abs(float(np.nanmedian(_all))) < 0.5, "the conventions do not tie out"

# the residual IS the predicted annuity-weighting term, with no free parameter
_ok = PANEL[["ca_synthetic_bp", "ca_synthetic_pred_bp"]].dropna()
print(f"corr(control, no-free-parameter prediction) = "
      f"{np.corrcoef(_ok['ca_synthetic_bp'], _ok['ca_synthetic_pred_bp'])[0,1]:.4f}")
_res = (_ok["ca_synthetic_bp"] - _ok["ca_synthetic_pred_bp"])
print(f"control minus prediction: mean {_res.mean():+.4f}bp  sd {_res.std():.4f}bp")

# the annual/quarterly gap is a PREDICTION: 0.375 * r^2, no free parameter
from RVUtils.ConvexityRV.ca_diagnostics import (COMPOUNDING_GAP_SLOPE,
                                                regress_gap_on_rate_squared)

_g = regress_gap_on_rate_squared(PANEL["annual_qq_gap_bp"], PANEL["swap_rate"])
print(f"\nannual-vs-Q/Q gap ~ b*r^2 (no intercept): b = {_g['slope_no_intercept']:.5f} "
      f"vs predicted {COMPOUNDING_GAP_SLOPE}  "
      f"({100*(_g['slope_no_intercept']/COMPOUNDING_GAP_SLOPE-1):+.2f}%), "
      f"r2 = {_g['r2']:.4f}, intercept {_g['intercept']:+.3f}bp, n={int(_g['n']):,}")
assert abs(_g["slope_no_intercept"] - COMPOUNDING_GAP_SLOPE) < 0.05

print("\nCAVEAT, stated rather than buried: at ranks >= 9 the swap curve carries")
print("~1 node per pack window and the four forwards spread only a few bp, so the")
print("control has LITTLE POWER there -- its ~0.00bp is close to an absence of")
print("evidence. It confirms the conventions where it CAN see (ranks 1-8, forward")
print("spreads of 16-117bp); for the deep windows the weight is carried by the")
print("per-contract settle-agreement gate in section 6, not by this control.")

zero-convexity control by rank (bp):
         n  ca_syn_med  ca_syn_p95  pred_med  swap_spread_bp  swap_nodes_in  annual_gap_bp  resid_after_pred
rank                                                                                                        
1     2068     -0.0434      0.9403   -0.0002         37.1306           11.0         3.8006            0.0001
2     2068     -0.0536      0.6448   -0.0330         28.5171            9.0         3.5256            0.0000
3     2067     -0.0502      0.6106   -0.0195         21.3898            7.0         3.4667            0.0000
4     2066     -0.0228      0.4355   -0.0141         15.6003            5.0         3.3273            0.0000
5     2066     -0.0184      0.4108   -0.0101         16.5467            4.0         3.1410            0.0003
6     2066     -0.0332      0.4152   -0.0004         19.6882            3.0         3.0514            0.0005
7     2065     -0.0272      0.3715    0.0001         15.3667            2.0         2.9781 

In [12]:
_f = go.Figure()
_f.add_trace(go.Scatter(x=_ctl.index, y=_ctl["swap_spread_bp"], name="swap-curve forward spread (bp)",
                        mode="lines+markers", line=dict(color="#4C78A8", width=3)))
_f.add_trace(go.Scatter(x=_ctl.index, y=_ctl["ca_syn_p95"].abs(),
                        name="|control residual| p95 (bp)", mode="lines+markers",
                        line=dict(color="#E45756", width=3), yaxis="y2"))
_f.update_layout(
    title="The zero-convexity control loses power exactly where the deep packs live"
          "<br><sub>USD-SOFR-1D thins to ~annual nodes past 2y, so a deep window's four "
          "quarterly forwards barely differ and the control returns ~0 by construction</sub>",
    xaxis_title="pack rank (first contract)",
    yaxis_title="control power: forward spread (bp)",
    yaxis2=dict(title="|residual| p95 (bp)", overlaying="y", side="right"),
    height=440)
_f

## 8. Known-answer tie-out — Citi Figure 58, close 6/9/2023

13 rows, windows 5..17, **including Blues (15.40bp) and Golds (22.29bp)** —
the rows the rank-2..10 strategy can never reach. All 24 contracts are cached
on this date, so both rate sources are available and the comparison is clean.

In [13]:
_depth = DEPTHS[NB.citi_date]
_n0 = network_calls_blocked()
_rows = Q.day_rows(NB.citi_date, _depth, cfg=QCFG, s2cfg=CITI)
assert network_calls_blocked() == _n0, "the tie-out reached for the network"
_t = pd.DataFrame(_rows)
_t = _t[(_t["rank"] >= 5) & (_t["rank"] <= 17)].sort_values("rank")

_out = []
for _, _r in _t.iterrows():
    _c = Q.CITI_SOFR_20230609.get(_r["pack"])
    if _c is None:
        continue
    _w = float(_r["time_weight"])
    _out.append({
        "rank": int(_r["rank"]), "pack": _r["pack"], "colour": _r["colour"] or "",
        "citi_CA": _c["ca_bp"],
        "settle_CA": round(float(_r["ca_bp_settle"]), 2),
        "q20_CA": round(float(_r["ca_bp_q20"]), 2),
        "d_settle": round(float(_r["ca_bp_settle"]) - _c["ca_bp"], 2),
        "d_q20": round(float(_r["ca_bp_q20"]) - _c["ca_bp"], 2),
        "citi_IV": _c["implied_vol_bp"],
        "IV_from_citi_CA": round(implied_vol_from_ca_bp(_c["ca_bp"], [math.sqrt(_w)]), 1),
        "IV_q20": round(implied_vol_from_ca_bp(float(_r["ca_bp_q20"]), [math.sqrt(_w)]), 1),
        "nodes_in": int(_r["q20_n_nodes_inside"]),
        "maxdiff_bp": round(float(_r["max_settle_diff_bp"]), 2),
        "gate": bool(_r["gate_ok"]),
    })
TIEOUT = pd.DataFrame(_out)
print(TIEOUT.to_string(index=False))

assert len(TIEOUT) == 13, f"expected Citi's 13 rows, got {len(TIEOUT)}"
for _src in ("d_q20", "d_settle"):
    _v = TIEOUT[_src].to_numpy(float)
    _lv = TIEOUT["q20_CA" if _src == "d_q20" else "settle_CA"].to_numpy(float)
    _corr = float(np.corrcoef(TIEOUT["citi_CA"], _lv)[0, 1])
    print(f"\n{_src}: mean {_v.mean():+.3f}  median {np.median(_v):+.3f}  "
          f"sd {_v.std(ddof=1):.3f}  |max| {np.abs(_v).max():.3f}  corr {_corr:.4f}")
    assert np.abs(_v).max() < NB.tieout_max_abs_bp, f"{_src} worst row {np.abs(_v).max():.2f}bp"
    assert _corr > NB.tieout_min_corr

_iv = (TIEOUT["IV_from_citi_CA"] / TIEOUT["citi_IV"])
print(f"\nimplied-vol inversion of Citi's OWN CA vs Citi's OWN IV column: "
      f"median ratio {_iv.median():.4f} (target {NB.iv_ratio_target}), "
      f"range {_iv.min():.4f}-{_iv.max():.4f}")
assert abs(float(_iv.median()) - NB.iv_ratio_target) < 0.004
assert bool(TIEOUT["gate"].all()), "every Citi row should pass the gate on this date"

print("\n*** THE PAYOFF ***")
for _p, _n in (("M6-H7", "BLUES"), ("M7-H8", "GOLDS")):
    _r = TIEOUT[TIEOUT["pack"] == _p].iloc[0]
    print(f"  {_n:6s} {_p}: Citi {_r['citi_CA']:6.2f}bp   ours(Q20) {_r['q20_CA']:6.2f}bp   "
          f"delta {_r['d_q20']:+.2f}bp   [rank {int(_r['rank'])}, unreachable at rank<=10]")

SUCCESS: `conv_tol` reached after 5 iterations (levenberg_marquardt), `f_val`: 0.012427280462850401, `time`: 0.0818s
 rank  pack colour  citi_CA  settle_CA  q20_CA  d_settle  d_q20  citi_IV  IV_from_citi_CA  IV_q20  nodes_in  maxdiff_bp  gate
    5 M4-H5   Reds     4.03       3.49    4.04     -0.54   0.01    199.5            198.3   198.6         7        1.32  True
    6 U4-M5            4.41       6.48    6.89      2.07   2.48    178.1            177.1   221.4         7        1.32  True
    7 Z4-U5            5.16       8.28    8.22      3.12   3.06    167.7            167.1   210.8         8        0.61  True
    8 H5-Z5            6.10       8.73    8.73      2.63   2.63    161.6            161.0   192.6         7        0.61  True
    9 M5-H6 Greens     8.24       7.70    7.57     -0.54  -0.67    168.5            167.9   161.0         7        0.56  True
   10 U5-M6            9.77       8.07    7.94     -1.70  -1.83    166.3            165.9   149.5         7        0.56  True
 

## 9. The rate-source control

The whole design demands this: if the gate works, the Q20 forward **is** the
settlement mark, and the two adjustments must agree to well inside a basis
point. A material divergence would mean the gate failed, not that the curve is
adding information.

In [14]:
_deep = GATED[GATED["rank"] >= NB.deep_rank_start]
_d = (_deep["ca_bp_q20"] - _deep["ca_bp_settle"]).abs()
print(f"gate-passed, ranks >= {NB.deep_rank_start} (n={len(_deep):,}): "
      f"median |CA_q20 - CA_settle| = {_d.median():.4f}bp, p95 = {_d.quantile(.95):.4f}bp, "
      f"corr = {np.corrcoef(_deep['ca_bp_q20'], _deep['ca_bp_settle'])[0,1]:.5f}")
assert float(_d.median()) < 0.1 and float(_d.quantile(.95)) < 1.0

_by = GATED.groupby("rank").agg(
    n=("ca_diff_bp", "size"), CA_q20=("ca_bp_q20", "median"), CA_settle=("ca_bp_settle", "median"),
    absdiff_med=("ca_diff_bp", lambda s: float(np.nanmedian(np.abs(s)))),
    absdiff_p95=("ca_diff_bp", lambda s: float(np.nanpercentile(np.abs(s), 95))))
_by["corr"] = [float(np.corrcoef(x["ca_bp_q20"], x["ca_bp_settle"])[0, 1])
               for _, x in GATED.groupby("rank")]
print("\nby rank (gate-passed):")
print(_by.round(4).to_string())

gate-passed, ranks >= 9 (n=14,443): median |CA_q20 - CA_settle| = 0.0484bp, p95 = 0.7225bp, corr = 0.99955

by rank (gate-passed):
         n  CA_q20  CA_settle  absdiff_med  absdiff_p95    corr
rank                                                           
1     1146  0.0969     0.0773       0.0569       0.2274  0.9943
2     1189  0.1923     0.1583       0.1374       0.5343  0.9764
3     1530  0.5045     0.4645       0.1026       0.3943  0.9928
4     1573  0.7476     0.7203       0.0927       0.4751  0.9925
5     1594  0.9215     0.9016       0.0945       0.5045  0.9973
6     1651  1.1805     1.2076       0.0770       0.3875  0.9994
7     1710  2.0057     1.9743       0.0661       0.3459  0.9996
8     1761  2.7322     2.7495       0.0585       0.3101  0.9995
9     1788  3.6859     3.7305       0.0567       0.2740  0.9989
10    1824  4.3660     4.4052       0.0445       0.2420  0.9997
11    1777  5.2416     5.2743       0.0378       0.1824  0.9998
12    1785  6.3861     6.4238       0

## 10. Why deep packs are the right place to trade this

The core quantitative argument, and it is a ratio rather than a level.
Barchart's EOD stamp is ~2h after the CME settle, and the resulting CA noise is
estimated two ways: the **Roll / MA(1)** estimator `σ_e = sqrt(−cov(Δ_t, Δ_{t−1}))`
(sharp) and `sd(Δ)/√2` (the upper bound, which assumes *all* variance is noise).

The noise barely grows with maturity while the adjustment grows four-fold —
so the same absolute measurement error is a third as damaging at Blues as at
Whites.

In [15]:
_g = GATED[GATED["year"].isin([2021, 2022, 2023])]
_rows = []
for _rk, _gr in _g.groupby("rank"):
    _w = _gr.pivot_table(index="date", columns="pack", values="ca_bp_q20", aggfunc="last")
    _dd = _w.diff()
    _roll, _ub, _sd, _lv, _tw = [], [], [], [], []
    for _c in _w.columns:
        _s = _dd[_c].dropna()
        if len(_s) < 40:
            continue
        _v = _s.var(ddof=1)
        _cov = _s.autocorr(lag=1) * _v
        _roll.append(math.sqrt(-_cov) if _cov < 0 else np.nan)
        _ub.append(math.sqrt(_v / 2)); _sd.append(math.sqrt(_v))
        _lv.append(_w[_c].dropna().median())
        _tw.append(_gr.loc[_gr["pack"] == _c, "time_weight"].median())
    if not _sd:
        continue
    _se, _sdm, _lvm = np.nanmedian(_roll), np.nanmedian(_sd), np.nanmedian(_lv)
    _M = np.nanmedian(_tw)
    _sig = implied_vol_from_ca_bp(_lvm, [math.sqrt(_M)]) if _lvm > 0 else np.nan
    _ds = vol_sensitivity_bp_per_bp(_lvm, [math.sqrt(_M)]) if _lvm > 0 else np.nan
    _rows.append({"rank": int(_rk), "T1_yrs": round(float(_gr["t1_first"].median()), 2),
                  "CA_level_bp": _lvm, "noise_roll_bp": _se, "noise_ub_bp": np.nanmedian(_ub),
                  "true_move_bp": math.sqrt(max(_sdm**2 - 2*_se**2, 0.0)) if np.isfinite(_se) else np.nan,
                  "noise/level %": 100*_se/_lvm if _lvm > 0 else np.nan,
                  "sigma_bp": _sig, "dsigma_dCA": _ds,
                  "vol_err %": 100*_ds*_se/_sig if np.isfinite(_ds) and _sig else np.nan})
NOISE = pd.DataFrame(_rows)
print(NOISE.round(3).to_string(index=False))
_near = NOISE[NOISE["rank"].between(1, 8)]
_dp = NOISE[NOISE["rank"] >= 13]
print(f"\nranks 1-8  : noise/level {_near['noise/level %'].median():5.1f}%   "
      f"CA-implied vol error {_near['vol_err %'].median():5.1f}%")
print(f"ranks 13-17: noise/level {_dp['noise/level %'].median():5.1f}%   "
      f"CA-implied vol error {_dp['vol_err %'].median():5.1f}%")
assert _dp["noise/level %"].median() < _near["noise/level %"].median(), (
    "the deep-pack case rests on this ratio falling with maturity")

 rank  T1_yrs  CA_level_bp  noise_roll_bp  noise_ub_bp  true_move_bp  noise/level %  sigma_bp  dsigma_dCA  vol_err %
    1    0.13       -0.210          0.142        0.208         0.214            NaN       NaN         NaN        NaN
    2    0.39       -0.045          0.186        0.227         0.184            NaN       NaN         NaN        NaN
    3    0.63        1.024          1.281        1.240         0.000        125.139   137.587      67.213     62.570
    4    0.88        1.716          0.750        0.827         0.491         43.710   144.338      42.045     21.855
    5    1.13        2.640          0.927        0.885         0.000         35.128   150.416      28.485     17.564
    6    1.38        4.280          1.236        1.206         0.000         28.882   164.786      19.252     14.441
    7    1.63        5.189          1.552        1.526         0.000         29.904   159.323      15.351     14.952
    8    1.88        5.755          1.056        1.125         0

In [16]:
_f = go.Figure()
_f.add_trace(go.Bar(x=NOISE["rank"], y=NOISE["CA_level_bp"], name="CA level (bp)",
                    marker_color="#4C78A8"))
_f.add_trace(go.Bar(x=NOISE["rank"], y=NOISE["noise_roll_bp"],
                    name="settle-timing noise, Roll/MA(1) (bp)", marker_color="#E45756"))
_f.add_trace(go.Scatter(x=NOISE["rank"], y=NOISE["noise/level %"], name="noise / level (%)",
                        yaxis="y2", mode="lines+markers", line=dict(color="#54A24B", width=3)))
_f.update_layout(
    title="The signal grows with maturity; the measurement noise does not<br>"
          "<sub>2021–2023, gate-passed. Rank 13 = Blues, 17 = Golds</sub>",
    xaxis_title="pack rank (first contract)", yaxis_title="bp",
    yaxis2=dict(title="noise / level (%)", overlaying="y", side="right", rangemode="tozero"),
    barmode="group", height=460)
_f

## 11. Staleness on the raw settle panel

`ca_staleness` runs on the **price** panel, not the adjustment, because that is
where the defect lives. One trap, found and fixed here: its default reference
rule picks the column with the fewest unchanged prints, and on a *rolling*
strip that is a contract present on only 117 of 1,410 dates — so "the market
moved" is almost never established and nothing is flagged. An always-present
front-contract reference is supplied explicitly.

In [17]:
_S = SETTLES.copy()
_S["FRONT"] = 100.0 - (PANEL[PANEL["rank"] == 1].set_index("date")["pack_rate_settle"]
                       .reindex(_S.index))
_auto = ((_S.drop(columns="FRONT").diff().abs() < 1e-9).sum()).idxmin()
print(f"default (auto) reference would be {_auto}, present on "
      f"{int(_S[_auto].notna().sum())}/{len(_S)} dates -- unusable")
FLAGS = ca_staleness.flag_stale_prices(_S, reference="FRONT")
print("flag incidence:", {c: round(float(FLAGS[c].mean()), 5) for c in
                          ("repeat_price", "stale_run", "jump_after_stale", "any_flag")})
CATCHUP = sorted(pd.DatetimeIndex(FLAGS.loc[FLAGS["jump_after_stale"], "date"].unique()))
print(f"catch-up (accumulate-then-jump) dates: {len(CATCHUP)} "
      f"{[str(d.date()) for d in CATCHUP]}")
print("\nworst contracts:")
print(ca_staleness.staleness_summary(FLAGS).head(6).round(5).to_string())

default (auto) reference would be SR3M18, present on 33/2068 dates -- unusable
flag incidence: {'repeat_price': 0.00755, 'stale_run': 0.00056, 'jump_after_stale': 9e-05, 'any_flag': 0.00764}
catch-up (accumulate-then-jump) dates: 9 ['2020-04-06', '2021-02-25', '2023-01-05', '2023-09-27', '2024-04-30', '2024-12-30', '2025-01-10', '2025-07-29', '2026-04-06']

worst contracts:
          repeat_price  stale_run  jump_after_stale  any_flag     n
contract                                                           
SR3Z21         0.01596    0.00290           0.00048   0.01644  2068
SR3M25         0.01451    0.00193           0.00000   0.01451  2068
SR3H26         0.01161    0.00097           0.00048   0.01209  2068
SR3M24         0.01209    0.00097           0.00000   0.01209  2068
SR3H22         0.01209    0.00193           0.00000   0.01209  2068
SR3M26         0.01161    0.00145           0.00000   0.01161  2068


### Verifying the detector before believing a near-zero answer

A checking tool that is itself broken reports success and hides the thing it
was built to find. So: inject a 5-print stale run and a 10bp catch-up into a
real contract, on a date **where the reference actually moved** — a repeat
print on a day the market did not move is not staleness, and in ZIRP the front
contract moves <0.5bp for months, which is how the first attempt at this check
silently "passed".

In [18]:
_col = _S.notna().sum().drop("FRONT").idxmax()
_mut = _S.copy()
_moved = (_mut["FRONT"].diff().abs() * 100.0) >= 0.5
_live = _mut.index[_mut[_col].notna() & _moved]
_live = _live[(_live > pd.Timestamp("2022-03-01")) & (_live < pd.Timestamp("2022-10-01"))]
_idx = list(_mut.index[_mut[_col].notna()])
_i0 = _idx.index(_live[len(_live) // 2])
_frozen = _mut.loc[_idx[_i0], _col]
for _k in range(1, 6):
    _mut.loc[_idx[_i0 + _k], _col] = _frozen
_mut.loc[_idx[_i0 + 6], _col] = _frozen + 0.10
_fm = ca_staleness.flag_stale_prices(_mut, reference="FRONT")
_sub = _fm[(_fm["contract"] == _col) & (_fm["date"].isin(_idx[_i0:_i0 + 8]))]
print(f"injected into {_col} at {_idx[_i0].date()}: 5 repeats + a 10bp catch-up")
print(_sub.to_string(index=False))
assert _sub["stale_run"].sum() >= 3, "detector missed the injected stale run"
assert _sub["jump_after_stale"].sum() >= 1, "detector missed the injected catch-up"
print("\n=> the detector fires on injected staleness, so the near-zero rate on the")
print("   real panel is a real measurement: these deep SR3 settles are NOT stale.")

injected into SR3H25 at 2022-06-16: 5 repeats + a 10bp catch-up
      date contract  repeat_price  stale_run  jump_after_stale  any_flag
2022-06-16   SR3H25         False      False             False     False
2022-06-17   SR3H25          True      False             False      True
2022-06-21   SR3H25          True       True             False      True
2022-06-22   SR3H25          True       True             False      True
2022-06-23   SR3H25          True       True             False      True
2022-06-24   SR3H25          True       True             False      True
2022-06-27   SR3H25         False      False              True      True
2022-06-28   SR3H25         False      False             False     False

=> the detector fires on injected staleness, so the near-zero rate on the
   real panel is a real measurement: these deep SR3 settles are NOT stale.


## 12. The backtest

Windows 9..14 (Greens through one past Blues), the **most recent contiguous
gate-passed run long enough to carry the 252-day z-score** — see `prep` below
for why "most recent" alone is a trap. Four runs: the Q20 rate source and the
raw-settle control, each hedged and unhedged.

**The book is marked off settles in every run.** The screen may be computed off
Q20 forwards, but a settle is what you can transact at and a curve forward is
not.

In [19]:
# A contiguous run shorter than the z-score's own warm-up cannot produce a
# signal -- `vs_model_z1y` is NaN on every day of it, `plan_epochs` skips every
# rebalance mark as warm-up, and the backtest marks a flat book. Runs below this
# floor are therefore not selectable at all; see `trim_to_contiguous_run`.
MIN_RUN_DAYS = DEEP.min_history_for_z1y
assert NEAR.min_history_for_z1y == MIN_RUN_DAYS, "both bands must share the floor"


def prep(lo, hi, source, keep="latest", min_days=MIN_RUN_DAYS):
    """Gate -> require every rank present -> keep one contiguous *viable* run.

    `keep="latest"` since 2026-08-19. The legacy `"longest"` rule cut this
    notebook's NEAR band from 518 dates to **301**, discarding every date in
    2023, 2024 and 2025 because the longest gap-free block happened to sit in
    2021 -- there it was the single largest killer in the whole pipeline, larger
    than the depth gate. Coverage is printed both ways below rather than one
    being chosen in silence.

    `min_days` since 2026-08-20, and it is the fix for a total, silent failure.
    Bare `"latest"` is `runs[-1]`, and on the rebuilt panel `runs[-1]` is a
    trailing island the store happened to warm last: **12 dates** for the deep
    band (2026-07-31..2026-08-18) and **9** for the near band, both cut off from
    2026-07-01 by an 18-day hole. Against `min_history_for_z1y` = 252 every
    `vs_model_z1y` on such a window is NaN, `plan_epochs` returns **zero**
    epochs, and the whole notebook died at `assert_ran` with `equity is
    identically zero -- no trigger fired` -- a message that reads as "the
    strategy made no money". Nothing upstream of it printed a warning.

    Note what this does NOT do: it touches no gate. `apply_gate`, the 2.0bp
    settle-agreement threshold and require-every-rank are all unchanged, and no
    date is admitted that was not already admitted. It only refuses to *select*
    a window too short to carry the statistic.
    """
    sub = Q.apply_gate(PANEL)
    sub = sub[(sub["rank"] >= lo) & (sub["rank"] <= hi)].copy()
    _full = sub.groupby("date")["rank"].nunique()
    sub = sub[sub["date"].isin(_full[_full == (hi - lo + 1)].index)]
    sub["ca_bp"] = sub[f"ca_bp_{source}"]
    sub["pack_rate"] = sub[f"pack_rate_{source}"]
    return S2.trim_to_contiguous_run(
        sub, RATES.loc[RATES.index.isin(sub["date"].unique())], keep=keep,
        min_days=min_days)


_lo, _hi = DEEP.rank_start - 1, DEEP.rank_start + DEEP.n_packs - 1
# Every run in the deep band, so the choice below is argued rather than assumed.
print(S2.coverage_by_run(prep(_lo, _hi, "q20", keep="none", min_days=0)[0])
      .to_string(), "\n")
for _src in ("q20", "settle"):
    for _keep in ("longest", "latest", "none"):
        for _floor in (0, MIN_RUN_DAYS):
            _p, _r = prep(_lo, _hi, _src, keep=_keep, min_days=_floor)
            _dd = pd.DatetimeIndex(sorted(_p["date"].unique()))
            _mark = " <- USED" if (_keep == "latest" and _floor == MIN_RUN_DAYS) else ""
            print(f"deep/{_src:6s} keep={_keep:<7s} min_days={_floor:<3d}: "
                  f"{len(_p):,} rows, {len(_dd)} days "
                  f"{_dd[0].date()}..{_dd[-1].date()}{_mark}")
DEEP_PANEL, DEEP_RATES = prep(_lo, _hi, "q20")
DEEP_DAYS = pd.DatetimeIndex(sorted(DEEP_PANEL["date"].unique()))
SPAN_YEARS = (DEEP_DAYS[-1] - DEEP_DAYS[0]).days / 365.25
print(f"\nspan_years = {SPAN_YEARS:.2f} (passed explicitly to every analytic below)")

        start         end  n_days  span_days  n_rows
0  2018-12-18  2018-12-18       1          0       7
1  2019-06-20  2023-02-01     743       1322    5201
2  2023-03-01  2023-07-26      21        147     147
3  2023-08-16  2024-11-01     197        443    1379
4  2024-12-19  2026-07-01     362        559    2534
5  2026-07-31  2026-08-18      12         18      84 

deep/q20    keep=longest min_days=0  : 5,201 rows, 743 days 2019-06-20..2023-02-01


deep/q20    keep=longest min_days=252: 5,201 rows, 743 days 2019-06-20..2023-02-01
deep/q20    keep=latest  min_days=0  : 84 rows, 12 days 2026-07-31..2026-08-18
deep/q20    keep=latest  min_days=252: 2,534 rows, 362 days 2024-12-19..2026-07-01 <- USED
deep/q20    keep=none    min_days=0  : 9,352 rows, 1336 days 2018-12-18..2026-08-18
deep/q20    keep=none    min_days=252: 9,352 rows, 1336 days 2018-12-18..2026-08-18
deep/settle keep=longest min_days=0  : 5,201 rows, 743 days 2019-06-20..2023-02-01
deep/settle keep=longest min_days=252: 5,201 rows, 743 days 2019-06-20..2023-02-01
deep/settle keep=latest  min_days=0  : 84 rows, 12 days 2026-07-31..2026-08-18
deep/settle keep=latest  min_days=252: 2,534 rows, 362 days 2024-12-19..2026-07-01 <- USED
deep/settle keep=none    min_days=0  : 9,352 rows, 1336 days 2018-12-18..2026-08-18
deep/settle keep=none    min_days=252: 9,352 rows, 1336 days 2018-12-18..2026-08-18

span_years = 1.53 (passed explicitly to every analytic below)


In [20]:
_EQF = DATA / "strat2_q20_equity.parquet"
if _EQF.exists():
    EQ = pd.read_parquet(_EQF)
    EQ.index = pd.to_datetime(EQ.index)
    print(f"loaded cached equity {EQ.shape}")
else:
    _series, _meta = {}, {}
    for _src in ("q20", "settle"):
        _p, _r = prep(_lo, _hi, _src)
        _days = pd.DatetimeIndex(sorted(_p["date"].unique()))
        _ts, _mo = S2.panel_timeseries(_p, DEEP), S2.model_timeseries(_p, DEEP)
        _specs = S2.plan_epochs(_p, _r, DEEP, ts=_ts, model=_mo, verbose=False)
        for _h in (False, True):
            _t = time.time()
            with cache_only():
                _bt = S2.run_backtest(_specs, DEEP, hedged=_h, trading_days=_days,
                                      name=f"deep_{_src}", show_progress=False)
            _e = S2.assert_ran(_bt, _specs, hedged=_h, expect_days=len(_days))
            _k = f"deep_{_src}_{'hedged' if _h else 'unhedged'}"
            _series[_k] = _e
            _meta[_k] = {"epochs": len(_specs),
                         "packs": sorted({s.pack for s in _specs}),
                         "ranks": sorted({s.rank for s in _specs})}
            print(f"  {_k:26s} {len(_e)} marks in {time.time()-_t:.0f}s, "
                  f"terminal {float(_e.iloc[-1]):+,.0f}")
    EQ = pd.DataFrame(_series)
    EQ.to_parquet(_EQF)
    import json

    (DATA / "strat2_q20_epochs.json").write_text(json.dumps(_meta, indent=2))

assert float(EQ.abs().max().max()) > 0, "equity identically zero -- no trigger fired"
print(EQ.tail(2).round(0).to_string())

C:\Users\chris\anaconda3\envs\stir\Lib\site-packages\rateslib\data\fixings.py:3426: RuntimeWarning:

invalid value encountered in divide



  deep_q20_unhedged          362 marks in 9s, terminal +191,174


  deep_q20_hedged            362 marks in 9s, terminal +12,944


  deep_settle_unhedged       362 marks in 7s, terminal +334,280


  deep_settle_hedged         362 marks in 10s, terminal +245,303
            deep_q20_unhedged  deep_q20_hedged  deep_settle_unhedged  deep_settle_hedged
2026-06-29           318016.0         160694.0              461122.0            392342.0
2026-07-01           191174.0          12944.0              334280.0            245303.0


### Near packs on the SAME panel and the SAME gate

The shipped near-pack result (windows 2..10, raw settles) is
**2020-08-03..2024-05-08, unhedged +$5,054,411 Sharpe 0.470 / hedged
+$4,448,142 Sharpe 0.409**. That window is not this one, and most of its P&L
comes from the 2023–24 narrowing that lies outside the deep window entirely.
So it is quoted as the reference and the near strategy is *also* re-run here on
the gate this notebook applies, which is a much shorter window — itself a
finding: the gate rejects a large share of near-pack rows.

In [21]:
_NEQF = DATA / "strat2_q20_equity_near.parquet"
if _NEQF.exists():
    EQN = pd.read_parquet(_NEQF)
    EQN.index = pd.to_datetime(EQN.index)
else:
    _p, _r = prep(NEAR.rank_start - 1, NEAR.rank_start + NEAR.n_packs - 1, "settle")
    _days = pd.DatetimeIndex(sorted(_p["date"].unique()))
    _specs = S2.plan_epochs(_p, _r, NEAR, verbose=False)
    _s = {}
    for _h in (False, True):
        with cache_only():
            _bt = S2.run_backtest(_specs, NEAR, hedged=_h, trading_days=_days,
                                  name="near_settle", show_progress=False)
        _s[f"near_settle_{'hedged' if _h else 'unhedged'}"] = S2.assert_ran(
            _bt, _specs, hedged=_h, expect_days=len(_days))
    EQN = pd.DataFrame(_s)
    EQN.to_parquet(_NEQF)
_nd = EQN.index
print(f"near packs, gated: {len(EQN)} days {_nd[0].date()}..{_nd[-1].date()}")

near packs, gated: 381 days 2024-10-04..2026-07-01


## 13. Results

In [22]:
def as_book(eq: pd.Series) -> pd.DataFrame:
    d = eq.diff().dropna()
    return pd.DataFrame({"timestamp": d.index, "pnl": d.to_numpy(float)})


BOOKS = {k: as_book(EQ[k]) for k in EQ.columns}
BOOKS_NEAR = {k: as_book(EQN[k]) for k in EQN.columns}


def _stats(name, eq, drop=()):
    d = eq.diff().dropna()
    keep = ~d.index.normalize().isin(pd.DatetimeIndex(drop))
    d2 = d[keep]
    sh = lambda x: float(x.mean() / x.std(ddof=1) * math.sqrt(252)) if x.std(ddof=1) > 0 else np.nan
    return {"run": name, "days": len(eq), "start": eq.index[0].date(), "end": eq.index[-1].date(),
            "pnl": float(eq.iloc[-1]), "sharpe": sh(d),
            "maxDD": float((eq - eq.cummax()).min()),
            "pnl_ex_catchup": float(d2.sum()), "sharpe_ex_catchup": sh(d2),
            "n_dropped": int((~keep).sum())}


RESULTS = pd.DataFrame([_stats(k, EQ[k], CATCHUP) for k in EQ.columns]
                       + [_stats(k, EQN[k], CATCHUP) for k in EQN.columns])
print(RESULTS.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

                 run  days      start        end         pnl  sharpe          maxDD  pnl_ex_catchup  sharpe_ex_catchup  n_dropped
   deep_q20_unhedged   362 2024-12-19 2026-07-01 191,174.232   0.079   -600,930.146      99,379.270              0.042          4
     deep_q20_hedged   362 2024-12-19 2026-07-01  12,943.879   0.005   -766,946.375     -57,188.846             -0.023          4
deep_settle_unhedged   362 2024-12-19 2026-07-01 334,280.443   0.123   -671,131.673     242,485.480              0.090          4
  deep_settle_hedged   362 2024-12-19 2026-07-01 245,303.357   0.088   -844,342.118     173,934.180              0.063          4
near_settle_unhedged   381 2024-10-04 2026-07-01 297,440.189   0.089 -1,181,769.216     215,170.636              0.064          3
  near_settle_hedged   381 2024-10-04 2026-07-01 281,948.217   0.084 -1,181,769.216     186,695.497              0.056          3


In [23]:
fig_cmp = compare_curves(BOOKS, title="Strat 2 deep packs (windows 9–14, incl. Blues): "
                                      f"Q20 forwards vs raw settles, hedged vs unhedged "
                                      f"($100k DV01, {SPAN_YEARS:.1f}y)")
fig_cmp

In [24]:
fig_dash = trade_dashboard(BOOKS["deep_q20_unhedged"],
                           title="Deep packs, Q20 rate source — unhedged (daily marks)",
                           span_years=SPAN_YEARS)
fig_dash

In [25]:
fig_dash_h = trade_dashboard(BOOKS["deep_q20_hedged"],
                             title="Deep packs, Q20 rate source — hedged with the 2s5s10s fly",
                             span_years=SPAN_YEARS)
fig_dash_h

In [26]:
for _k in EQ.columns:
    print(f"--- {_k}")
    print(summary_stats(BOOKS[_k], span_years=SPAN_YEARS).to_string(index=False))

--- deep_q20_unhedged
                metric                     value
                trades                       361
           net / trade                 +529.5685
         gross / trade                         -
          cost / trade                         -
             total net                +191174.23
              hit rate                     15.5%
    avg win / avg loss  +144384.331 / -25883.109
          payoff ratio                     5.578
        Sharpe / trade                    0.0050
           t-statistic                      0.10
    best / worst trade +495842.018 / -503468.545
          max drawdown                -600930.15
longest win / loss run                   3 / 251
              avg hold                         -
         trades / year                     235.9
     annualised Sharpe                    0.0769
--- deep_q20_hedged
                metric                     value
                trades                       361
           net / trade     

## 14. Selection stability — the honest caveat

The two rate sources agree on the *level* of the adjustment to 0.03bp and
correlate at 1.0000. They nonetheless produce materially different P&L, and the
reason is worth stating plainly: the trade is chosen by a **cross-sectional
ranking of nearly identical numbers**, so a 0.03bp difference can flip which
pack is top-ranked and change an entire epoch.

This is a statement about the strategy's fragility, not about the rate source's
accuracy — and it bounds how much any of the P&L numbers above should be
trusted to two significant figures.

In [27]:
_pa, _ra = prep(_lo, _hi, "q20")
_pb, _rb = prep(_lo, _hi, "settle")
_tsa, _ma = S2.panel_timeseries(_pa, DEEP), S2.model_timeseries(_pa, DEEP)
_tsb, _mb = S2.panel_timeseries(_pb, DEEP), S2.model_timeseries(_pb, DEEP)
_common = sorted(set(_pa["date"]).intersection(_pb["date"]))
_same = _tot = 0
for _d in _common[::5]:
    _sa = S2.daily_screen(pd.Timestamp(_d).date(), _pa, DEEP, ts=_tsa, model=_ma)
    _sb = S2.daily_screen(pd.Timestamp(_d).date(), _pb, DEEP, ts=_tsb, model=_mb)
    if _sa.empty or _sb.empty:
        continue
    _ka, _ = S2.select_pack(_sa, DEEP)
    _kb, _ = S2.select_pack(_sb, DEEP)
    _tot += 1
    _same += int(_ka == _kb)
print(f"same pack selected on {_same}/{_tot} screen days ({100*_same/max(_tot,1):.1f}%)")
print("=> the ~10% of days where the two disagree is what produces the P&L gap above.")

same pack selected on 52/73 screen days (71.2%)
=> the ~10% of days where the two disagree is what produces the P&L gap above.


## 15. What would make this wrong

Ranked by size of the error each could introduce, all measured rather than
asserted.

| # | risk | size | status |
|---|------|------|--------|
| 1 | **Node degeneracy** — a pack window inside one log-linear segment. The zero-convexity control is structurally blind to it and `CA≥0` / `corr(CA,T1²)` *reward* it. | unbounded; 15.8bp median at 2019 ranks 1–4 | **gated out** (`spans_window`, `n_nodes_inside`) |
| 2 | **Wrong futures source.** The production builder fetches from `BARCHART_TOS_LIVE_STIRF-RL` (22:xx intraday, depth 20 on zero local dates), not the 17:00 EOD settle. | would be a 52–57-request-per-date crawl AND the wrong mark | **fixed by injection**; guard asserts 0 requests |
| 3 | **Settle-timing.** Barchart "EOD" is ~2h after the CME settle. | 0.6–1.6bp/pack-day; **30.1%** of the CA level at ranks 1–8, **6.7%** at 13–17 | irreducible; the reason to trade deep |
| 4 | **Selection fragility.** A 0.03bp CA difference flips the ranked winner on ~10% of screen days. | ~2× on total P&L | **reported, not fixed** — the dominant caveat |
| 5 | **IMM roll off-by-one.** The `SFRCM` ladder rolled on the IMM date; the pack universe did not. | 8 of 10 network reaches in a full build | **fixed at source** 2026-08-19 (`tos._imm_cutoff` keeps the contract; the `instrument_count` shim was deleted with it) |
| 6 | **Swap-leg frequency.** `usd_irs` quotes annual fixed; Citi specifies Q/Q. | up to 12bp (`0.375·r²`) | fixed upstream (`matched_forward_swap_rate`) |
| 7 | **Stale deferred settles.** | `stale_run` 0.055%, 4 catch-up dates; removing them moves deep hedged P&L by ~$0.8mn | **reported both ways** |
| 8 | **Universe truncation.** Golds needs depth 20 → 681 dates whose longest contiguous run (309 days) is entirely inside ZIRP. | Golds cannot be backtested daily | **documented gap** |

### The gap, stated plainly

Golds (rank 17) is **reproduced on the tie-out date to −0.56bp** but **cannot
be backtested**: it needs a 20-contract strip, which exists on 681 dates whose
longest gap-free run is 2019-09-13..2020-12-16 — 309 days, entirely inside
ZIRP, where the adjustment is ~0 and a 252-day z-score leaves ~57 tradeable
days. That is a data-coverage limit, not a modelling one; the code produces
Citi's exact 13 rows the moment the strip is warm, as section 8 demonstrates.

In [28]:
print("network calls blocked over the whole notebook:", network_calls_blocked())
print("\nCoverage summary")
print(f"  panel            {len(PANEL):,} rows x {PANEL['date'].nunique():,} dates")
print(f"  gate-passed      {len(GATED):,} rows ({100*len(GATED)/len(PANEL):.1f}%)")
print(f"  deep backtest    {len(DEEP_DAYS)} days {DEEP_DAYS[0].date()}..{DEEP_DAYS[-1].date()}")
print(f"  Blues reachable  {PANEL[PANEL['rank']==13]['date'].nunique():,} dates")
print(f"  Golds reachable  {PANEL[PANEL['rank']==17]['date'].nunique():,} dates")

network calls blocked over the whole notebook: 0

Coverage summary
  panel            32,540 rows x 2,068 dates
  gate-passed      26,597 rows (81.7%)
  deep backtest    362 days 2024-12-19..2026-07-01
  Blues reachable  1,807 dates
  Golds reachable  1,302 dates
